In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
pd.set_option('display.max_columns', 1000)

In [2]:
import os
from pyspark.sql import SparkSession
from pyspark.sql import Row
spark = SparkSession.builder \
    .appName("Split") \
    .config("spark.driver.memory", "10g") \
    .config("spark.executor.memory", "8g") \
    .getOrCreate()
spark.sparkContext.setLogLevel("ERROR")
spark.conf.set("spark.sql.shuffle.partitions", "400")
data_dir = "data/icd_sections"
csv_files = [f for f in os.listdir(data_dir) if f.endswith(".csv")]

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/25 17:34:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
rows_list = []

for csv_file in csv_files:
    csv_file_path = os.path.join(data_dir, csv_file)
    df = spark.read.csv(csv_file_path, header=True, inferSchema=True)
    
    num_rows = df.count()
    num_cols = len(df.columns)
    num_patients = df.select('Beneficiary_ID').distinct().count()
    
    rows_list.append(Row(
        file_name=csv_file,
        num_rows=num_rows,
        num_columns=num_cols,
        num_patients=num_patients
    ))
summary_df = spark.createDataFrame(rows_list)
summary_df.show(truncate=False)

+-----------------+--------+-----------+------------+
|file_name        |num_rows|num_columns|num_patients|
+-----------------+--------+-----------+------------+
|OASIS_U00-U85.csv|3       |300        |2           |
|OASIS_A00-B99.csv|1644414 |300        |352672      |
|OASIS_P00-P96.csv|2927    |300        |551         |
|OASIS_C00-D49.csv|1008147 |300        |276222      |
|OASIS_E00-E89.csv|3949248 |300        |881628      |
|OASIS_S00-T88.csv|141142  |300        |107581      |
|OASIS_G00-G99.csv|2154843 |300        |507180      |
|OASIS_K00-K95.csv|1749157 |300        |401060      |
|OASIS_Q00-Q99.csv|45428   |300        |10319       |
|OASIS_R00-R99.csv|186770  |300        |139090      |
|OASIS_H60-H95.csv|58820   |300        |15161       |
|OASIS_L00-L99.csv|2180040 |300        |455716      |
|OASIS_Z00-Z99.csv|152865  |300        |122114      |
|OASIS_N00-N99.csv|3472547 |300        |763582      |
|OASIS_O00-O9A.csv|6405    |300        |2250        |
|OASIS_H00-H59.csv|78069   |

In [4]:
summary_pd = summary_df.toPandas()
summary_pd.to_csv("data/icd_section_summary.csv", index=False)
